[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/orange/notebooks/orange_mtb_proteome_embeddings.ipynb)

# Compute ESM-C embeddings for the Mtb proteome

**Orange group · Tuberculosis**

A protein language model reads a protein sequence and turns it into a list of numbers (an *embedding*) that captures what the model has learned about its structure and function. Here we compute one embedding for each of the roughly 4,000 proteins of *Mycobacterium tuberculosis*, so that later notebooks can compare, cluster and prioritise candidate drug targets.

## What you will do

- Download the *M. tuberculosis* H37Rv reference proteome from UniProt.
- Load ESM-C 300M, a protein language model from EvolutionaryScale.
- Compute one embedding per protein, in batches that can be resumed.
- Save all embeddings in a single CSV file.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "orange"
NEEDS_GPU = True
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose T4 GPU, and run this cell again.")

## 1. Download the reference proteome

A *proteome* is the full set of proteins made by an organism. UniProt, the main public protein database, keeps a *reference proteome* for *M. tuberculosis* strain H37Rv, the laboratory strain most research uses. Its identifier is `UP000001584`.

The cell below downloads the proteome as a compressed FASTA file (a plain-text format for sequences). It only downloads it if it isn't already there.

In [ ]:
import urllib.request
from pathlib import Path

PROTEOME = "UP000001584"  # M. tuberculosis H37Rv reference proteome
url = f"https://rest.uniprot.org/uniprotkb/stream?query=proteome:{PROTEOME}&format=fasta&compressed=true"
fasta_path = Path("data/downloads/mtb_proteome.fasta.gz")
fasta_path.parent.mkdir(parents=True, exist_ok=True)
if not fasta_path.exists():
    urllib.request.urlretrieve(url, fasta_path)
print(fasta_path, round(fasta_path.stat().st_size / 1e6, 1), "MB")

Each protein in the FASTA file starts with a header line (`>sp|P9WGR1|...`) followed by its sequence. We read it into a table with the UniProt accession (`UniprotAC`, the protein's unique identifier), the gene name and the sequence. Sorting by accession keeps the order the same every time.

In [ ]:
import gzip
import re
import pandas as pd

records = []
with gzip.open(fasta_path, "rt") as f:
    for block in ("\n" + f.read()).split("\n>")[1:]:  # each protein starts with ">" on a new line
        header, *lines = block.splitlines()
        gene = re.search(r"GN=(\S+)", header)
        records.append({"UniprotAC": header.split("|")[1], "gene": gene.group(1) if gene else "", "sequence": "".join(lines)})
proteome = pd.DataFrame(records).sort_values("UniprotAC").reset_index(drop=True)
proteome["length"] = proteome["sequence"].str.len()
print(len(proteome), "proteins")
proteome.head()

## 2. Look at protein lengths

ESM-C can read at most about 2,000 amino acids at once. Most Mtb proteins are much shorter, but a few are longer. We check how many, because those need special handling in section 4.

The histogram shows how protein lengths are spread, and the printed number is how many proteins are longer than 2,000 amino acids.

In [ ]:
import stylia

# Format: slide | Style: ersilia — change with stylia.set_format() / stylia.set_style()
stylia.set_format("slide")
stylia.set_style("ersilia")
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.hist(proteome["length"], bins=100)
stylia.label(ax, xlabel="Protein length (amino acids)", ylabel="Number of proteins")
stylia.save_figure("data/downloads/protein_lengths.png")
print((proteome["length"] > 2000).sum(), "proteins longer than 2,000 amino acids")

## 3. Load ESM-C 300M

ESM-C (*ESM Cambrian*) is a protein language model trained on millions of natural protein sequences. We use the smallest version, with 300 million parameters, which runs on a laptop. For each amino acid it produces 960 numbers.

The cell below picks the fastest device available: an NVIDIA GPU (`cuda`, e.g. in Colab), an Apple GPU (`mps`) or the normal processor (`cpu`). It then downloads the model weights (about 1 GB, only the first time) and loads them.

In [ ]:
import numpy as np
import torch
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

torch.manual_seed(42)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
model = ESMC.from_pretrained("esmc_300m").to(device).eval()
print("Model loaded on", device)

## 4. Embed one protein

The model gives one embedding per amino acid. To get a single embedding for the whole protein, we take the average over all its amino acids (*mean pooling*). The model also adds a start and an end token to each sequence, which we leave out.

The function below does this. Proteins longer than 2,000 amino acids are cut into pieces of 2,000, each piece is embedded, and the pieces are combined in proportion to their length. We try it on the first protein.

In [ ]:
WINDOW = 2000

def embed(sequence):
    """Return one embedding for a protein: the average of its amino-acid embeddings."""
    total = 0
    for start in range(0, len(sequence), WINDOW):
        piece = ESMProtein(sequence=sequence[start:start + WINDOW])
        with torch.no_grad():
            output = model.logits(model.encode(piece), LogitsConfig(sequence=True, return_embeddings=True))
        total = total + output.embeddings[0, 1:-1].sum(dim=0)  # drop start and end tokens
    return (total / len(sequence)).float().cpu().numpy()

vector = embed(proteome["sequence"][0])
print(proteome["UniprotAC"][0], vector.shape, vector[:5])

## 5. Embed the whole proteome

Embedding about 4,000 proteins takes a while (around 15 minutes on a laptop GPU, longer on a CPU). To avoid losing work if something goes wrong, we split the proteins into batches of 250 and save each batch as its own file in `bigfiles/mtb_esmc300m_batches/`. Batches that are already saved are skipped, so if the run stops you just run this cell again and it continues where it left off.

> **Note:** A batch is first written under a temporary name and only renamed once it is complete, so a half-written file is never mistaken for a finished one. If you change `BATCH_SIZE`, delete the batch folder first, or the old and new batches will not match.

The cell below runs the batches and prints how many are finished.

In [ ]:
from tqdm.auto import tqdm

BATCH_SIZE = 250
batch_dir = Path("../../bigfiles/mtb_esmc300m_batches")
batch_dir.mkdir(parents=True, exist_ok=True)
columns = [f"dim_{i:03d}" for i in range(vector.shape[0])]
n_batches = (len(proteome) + BATCH_SIZE - 1) // BATCH_SIZE
for b in tqdm(range(n_batches), desc="Batches"):
    path = batch_dir / f"batch_{b:03d}.csv"
    if path.exists():
        continue  # already done in an earlier run
    chunk = proteome.iloc[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
    batch = pd.DataFrame(np.array([embed(s) for s in chunk["sequence"]]).round(4), columns=columns)
    batch.insert(0, "UniprotAC", chunk["UniprotAC"].values)
    batch.to_csv(path.with_suffix(".tmp"), index=False)
    path.with_suffix(".tmp").rename(path)
print(len(list(batch_dir.glob("batch_*.csv"))), "of", n_batches, "batches done")

## 6. Save the embeddings

Finally we join all batches into one table: the first column is `UniprotAC` and the other 960 columns (`dim_000` to `dim_959`) are the embedding. We check that every protein is there exactly once before saving it as `bigfiles/mtb_esmc300m_embeddings.csv`.

> **Note:** `bigfiles/` sits at the top of the repository and is not uploaded to GitHub, because the file is large (about 30 MB). In Colab it is deleted when the runtime disconnects, so download it first if you want to keep it.

The cell below combines the batches, checks them and saves the final CSV.

In [ ]:
embeddings = pd.concat([pd.read_csv(f) for f in sorted(batch_dir.glob("batch_*.csv"))], ignore_index=True)
assert embeddings["UniprotAC"].tolist() == proteome["UniprotAC"].tolist(), "Batches are missing or out of date: re-run section 5"
assert not embeddings.isna().any().any()
output_path = Path("../../bigfiles/mtb_esmc300m_embeddings.csv")
embeddings.to_csv(output_path, index=False)
print(output_path, embeddings.shape)
embeddings.head()

## Summary

- We downloaded the *M. tuberculosis* H37Rv reference proteome (`UP000001584`) from UniProt.
- We computed an ESM-C 300M embedding (960 numbers) for every protein, in batches that can be resumed.
- All embeddings are saved in `bigfiles/mtb_esmc300m_embeddings.csv`, one row per protein, with `UniprotAC` as the first column.

**Next:** use the embeddings to compare and cluster the shortlisted candidate targets, or as input features for models that predict druggability.